# Real-Time Object Detection and Tracking

Real-time object detection and tracking using:
- Live camera feed or video file
- Standard pipeline (background subtraction)
- SIFT-enhanced pipeline (feature validation)

Features: FPS display, live metrics, recording, keyboard controls, pipeline switching


In [41]:
import cv2
import numpy as np
import time
import sys
from collections import deque
from datetime import datetime
sys.path.append('..')

from sot import (
    Pipeline, PipelineStep,
    DetectionStep, TrackingStep, VisualizationStep,
    FramePreprocessingStep,
    SIFTFeatureStep, SIFTMatchingStep, 
    SIFTEnhancedDetectionStep, SIFTVisualizationStep
)
from detector import ObjectDetector
from tracker import ObjectTracker


## Configuration


In [42]:
USE_CAMERA = False
CAMERA_ID = 0
VIDEO_PATH = "../data/videos/bees_long.mp4"

USE_SIFT = True

WINDOW_NAME = "Real-Time Tracking"
DISPLAY_WIDTH = 1280
DISPLAY_HEIGHT = None

ENABLE_RECORDING = True
OUTPUT_DIR = "realtime_output"
OUTPUT_FPS = 15
FPS_SMOOTHING = 30
PROCESS_EVERY_N_FRAMES = 1


## Initialize Pipelines


In [ ]:
# Standard pipeline
pipeline_standard = Pipeline()
pipeline_standard.add_step(FramePreprocessingStep(blur_ksize=5))
pipeline_standard.add_step(DetectionStep(detector=ObjectDetector(
    history=500, dist_threshold=800, min_area=30, max_area=100000,
    erode_size=(3, 3), dilate_size=(8, 8)
)))
pipeline_standard.add_step(TrackingStep(tracker=ObjectTracker(
    max_distance=30.0, min_hits=3, max_age=30
)))
pipeline_standard.add_step(VisualizationStep(
    draw_detections=True, draw_tracks=True, draw_trajectories=True,
    draw_ids=True, draw_stats=True
))

pipeline_sift = Pipeline()
pipeline_sift.add_step(FramePreprocessingStep(blur_ksize=5))
pipeline_sift.add_step(SIFTFeatureStep(
    nfeatures=300, contrastThreshold=0.04, edgeThreshold=10
))
pipeline_sift.add_step(SIFTMatchingStep(ratio_threshold=0.75, min_match_count=4))
pipeline_sift.add_step(SIFTEnhancedDetectionStep(
    detector=ObjectDetector(
        history=500, dist_threshold=800, min_area=30, max_area=100000,
        erode_size=(3, 3), dilate_size=(8, 8)
    ),
    min_features_in_detection=2,
    feature_density_threshold=0.001
))
pipeline_sift.add_step(TrackingStep(tracker=ObjectTracker(
    max_distance=30.0, min_hits=3, max_age=30
)))
pipeline_sift.add_step(SIFTVisualizationStep(
    draw_keypoints=True, draw_matches=True, draw_detections=True, max_keypoints=50
))
pipeline_sift.add_step(VisualizationStep(
    draw_detections=False, draw_tracks=True, draw_trajectories=True,
    draw_ids=True, draw_stats=True
))

active_pipeline = pipeline_sift if USE_SIFT else pipeline_standard


## Real-Time Processing

In [44]:
import os

if ENABLE_RECORDING:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

cap = cv2.VideoCapture(CAMERA_ID if USE_CAMERA else VIDEO_PATH)

if not cap.isOpened():
    print(f"Error: Could not open {'camera ' + str(CAMERA_ID) if USE_CAMERA else VIDEO_PATH}")
else:
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    display_width = DISPLAY_WIDTH
    display_height = int(frame_height * (DISPLAY_WIDTH / frame_width))
    
    print(f"Source: {frame_width}x{frame_height}, Display: {display_width}x{display_height}")
    
    video_writer = None
    if ENABLE_RECORDING:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        pipeline_name = "sift" if USE_SIFT else "standard"
        output_path = os.path.join(OUTPUT_DIR, f"realtime_{pipeline_name}_{timestamp}.mp4")
        video_writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'),
                                       OUTPUT_FPS, (display_width, display_height))
        print(f"Recording: {output_path}")
    
    frame_count = 0
    processed_count = 0
    fps_queue = deque(maxlen=FPS_SMOOTHING)
    is_paused = False
    recording_enabled = ENABLE_RECORDING
    current_pipeline = active_pipeline
    pipeline_name = "SIFT" if USE_SIFT else "Standard"
    process_interval = PROCESS_EVERY_N_FRAMES
    start_time = time.time()
    
    print("Controls: q=quit, s=switch, r=record, p=pause, space=reset, +/-=speed\n")
    
    try:
        while True:
            if not is_paused:
                ret, frame = cap.read()
                if not ret:
                    break
                
                frame_count += 1
                
                if frame_count % process_interval == 0:
                    t = time.time()
                    context = current_pipeline.process(frame.copy(), {'frame_number': processed_count})
                    vis_frame = context.get('visualization', context.get('sift_visualization', frame))
                    
                    fps_queue.append(1.0 / (time.time() - t))
                    avg_fps = np.mean(fps_queue)
                    num_dets = context.get('num_detections', 0)
                    num_tracks = context.get('num_confirmed', 0)
                    processed_count += 1
                else:
                    vis_frame = frame
                    avg_fps = fps_queue[-1] if len(fps_queue) > 0 else 0
                    num_dets = num_tracks = 0
                
                vis_frame = cv2.resize(vis_frame, (display_width, display_height))
                
                # Overlay
                overlay = vis_frame.copy()
                cv2.rectangle(overlay, (0, 0), (350, 150), (0, 0, 0), -1)
                cv2.addWeighted(overlay, 0.5, vis_frame, 0.5, 0, vis_frame)
                
                cv2.putText(vis_frame, f"FPS: {avg_fps:.1f}", (10, 25),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                cv2.putText(vis_frame, f"Pipeline: {pipeline_name}", (10, 50),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(vis_frame, f"Frame: {frame_count}", (10, 75),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                if processed_count > 0:
                    cv2.putText(vis_frame, f"Detections: {num_dets}", (10, 100),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                    cv2.putText(vis_frame, f"Tracks: {num_tracks}", (10, 125),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                
                if recording_enabled and video_writer:
                    cv2.circle(vis_frame, (display_width - 30, 30), 10, (0, 0, 255), -1)
                if is_paused:
                    cv2.putText(vis_frame, "PAUSED", (display_width // 2 - 80, display_height // 2),
                               cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)
                
                cv2.imshow(WINDOW_NAME, vis_frame)
                
                if recording_enabled and video_writer:
                    video_writer.write(vis_frame)
            
            key = cv2.waitKey(1) & 0xFF
            
            if key == ord('q') or key == 27:
                break
            elif key == ord('s'):
                current_pipeline = pipeline_sift if current_pipeline == pipeline_standard else pipeline_standard
                pipeline_name = "SIFT" if current_pipeline == pipeline_sift else "Standard"
                print(f"Switched to {pipeline_name}")
            elif key == ord('r'):
                recording_enabled = not recording_enabled
                print(f"Recording: {'ON' if recording_enabled else 'OFF'}")
            elif key == ord('p'):
                is_paused = not is_paused
                print(f"{'Paused' if is_paused else 'Resumed'}")
            elif key == ord(' '):
                pipeline_standard.reset()
                pipeline_sift.reset()
                print("Reset")
            elif key in [ord('+'), ord('=')]:
                process_interval = min(process_interval + 1, 10)
                print(f"Process every {process_interval} frames")
            elif key in [ord('-'), ord('_')]:
                process_interval = max(process_interval - 1, 1)
                print(f"Process every {process_interval} frames")
        
        elapsed = time.time() - start_time
        print(f"\nProcessed {frame_count} frames in {elapsed:.1f}s ({processed_count/elapsed:.1f} FPS)")
        if video_writer:
            print(f"Saved: {output_path}")
    
    except KeyboardInterrupt:
        print("\nInterrupted")
    except Exception as e:
        print(f"\nError: {e}")
    finally:
        cap.release()
        if video_writer:
            video_writer.release()
        cv2.destroyAllWindows()


Source: 2052x1156, Display: 1280x721
Recording: realtime_output/realtime_sift_20251223_065603.mp4
Controls: q=quit, s=switch, r=record, p=pause, space=reset, +/-=speed


Processed 159 frames in 21.7s (7.3 FPS)
Saved: realtime_output/realtime_sift_20251223_065603.mp4


In [45]:
def quick_test(source=0, use_sift=False, record=False, display_width=1280):
    """Quick test function for real-time detection."""
    pipeline = pipeline_sift if use_sift else pipeline_standard
    cap = cv2.VideoCapture(source)
    
    if not cap.isOpened():
        print(f"Failed to open: {source}")
        return
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    display_height = int(height * (display_width / width))
    
    writer = None
    if record:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = f"quick_test_{timestamp}.mp4"
        writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'),
                                15, (display_width, display_height))
    
    fps_queue = deque(maxlen=30)
    frame_count = 0
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            t = time.time()
            context = pipeline.process(frame.copy(), {'frame_number': frame_count})
            vis_frame = context.get('visualization', context.get('sift_visualization', frame))
            vis_frame = cv2.resize(vis_frame, (display_width, display_height))
            
            fps_queue.append(1.0 / (time.time() - t))
            cv2.putText(vis_frame, f"FPS: {np.mean(fps_queue):.1f}", (10, 30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            cv2.imshow("Quick Test", vis_frame)
            if writer:
                writer.write(vis_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        if writer:
            writer.release()
        cv2.destroyAllWindows()
        print(f"Processed {frame_count} frames at {np.mean(fps_queue):.1f} FPS")

# Usage: quick_test(source=0) or quick_test(source="../data/videos/bees_short.mp4", use_sift=True)


In [47]:
import glob

video_files = glob.glob("../data/videos/*.mp4")

for video in video_files:
    name = video.split('/')[-1]
    cap = cv2.VideoCapture(video)
    if cap.isOpened():
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f"{name}: {w}x{h}, {fps:.0f}fps, {frames} frames")
        cap.release()


bees_long.mp4: 2052x1156, 60fps, 1200 frames
bees1_longer.mp4: 2052x1156, 60fps, 1200 frames
city_longer.mp4: 2030x1080, 60fps, 600 frames
bees_short.mp4: 2052x1156, 60fps, 840 frames
bees_medium.mp4: 2052x1156, 60fps, 480 frames
videoplayback.mp4: 1920x1080, 30fps, 387 frames
bees2_longer.mp4: 2052x1156, 60fps, 1500 frames


## Performance Analysis


In [48]:
import glob
import os

output_files = glob.glob(os.path.join(OUTPUT_DIR, "*.mp4"))

for filepath in sorted(output_files, reverse=True):
    name = os.path.basename(filepath)
    size = os.path.getsize(filepath) / (1024 * 1024)
    
    cap = cv2.VideoCapture(filepath)
    if cap.isOpened():
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration = frames / fps if fps > 0 else 0
        print(f"{name}: {w}x{h}, {fps:.0f}fps, {frames} frames, {duration:.1f}s, {size:.1f}MB")
        cap.release()


realtime_standard_20251223_065534.mp4: 1280x720, 15fps, 256 frames, 17.1s, 23.3MB
realtime_standard_20251223_065520.mp4: 1280x720, 45fps, 95 frames, 2.1s, 6.4MB
realtime_standard_20251223_065454.mp4: 1280x720, 45fps, 25 frames, 0.6s, 0.7MB
realtime_standard_20251223_065420.mp4: 1280x720, 15fps, 51 frames, 3.4s, 6.9MB
realtime_sift_20251223_065603.mp4: 1280x720, 15fps, 159 frames, 10.6s, 4.2MB
realtime_sift_20251223_065343.mp4: 1280x720, 15fps, 101 frames, 6.7s, 2.0MB
realtime_sift_20251223_064119.mp4: 1280x720, 15fps, 123 frames, 8.2s, 1.2MB


## Troubleshooting

**Camera not working:** Try different CAMERA_ID (0, 1, 2). Check camera permissions on macOS/Linux.

**Low FPS:** Reduce DISPLAY_WIDTH, increase PROCESS_EVERY_N_FRAMES, or switch to standard pipeline.

**High CPU:** Use standard pipeline, reduce nfeatures parameter, or disable recording.


## Notes

**Standard pipeline:** Fast, suitable for clean backgrounds and real-time applications.

**SIFT pipeline:** Better accuracy for textured objects, cluttered backgrounds. Slower (50-100% overhead).

**Tips:** Good lighting, stable camera, adequate contrast help both pipelines. Start with standard, switch to SIFT if needed.
